# CV Strategies on the Same Dataset

Three splitters, same data, **very different** stories. We'll simulate temporal & group structure on top of our synthetic data to make the point clearly.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, GroupKFold, TimeSeriesSplit
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

SEED = 42
rng = np.random.default_rng(SEED)
df = pd.read_parquet(Path('..') / 'data' / 'transactions.parquet').reset_index(drop=True)

# Simulate timestamps spanning 6 months and ~10k unique users with repeated sessions.
df['ts'] = pd.to_datetime('2025-01-01') + pd.to_timedelta(
    rng.integers(0, 180*24*3600, size=len(df)), unit='s'
)
df['user_id'] = rng.integers(0, 10_000, size=len(df))
df = df.sort_values('ts').reset_index(drop=True)

for c in ['device_type', 'country']:
    df[c] = df[c].astype('category')
X = df.drop(columns=['is_fraud', 'ts', 'user_id'])
y = df['is_fraud'].values
groups = df['user_id'].values
print(df.shape, df['ts'].min(), df['ts'].max())

In [ ]:
def score_cv(splitter, X, y, groups=None):
    aps = []
    iterator = splitter.split(X, y, groups) if groups is not None else splitter.split(X, y)
    for tr_idx, te_idx in iterator:
        m = XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            tree_method='hist', enable_categorical=True,
            eval_metric='aucpr', random_state=SEED,
        )
        m.fit(X.iloc[tr_idx], y[tr_idx])
        proba = m.predict_proba(X.iloc[te_idx])[:, 1]
        aps.append(average_precision_score(y[te_idx], proba))
    return np.array(aps)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
gkf = GroupKFold(n_splits=5)
tss = TimeSeriesSplit(n_splits=5)

scores_skf = score_cv(skf, X, y)
scores_gkf = score_cv(gkf, X, y, groups=groups)
scores_tss = score_cv(tss, X, y)

summary = pd.DataFrame({
    'StratifiedKFold (IID assumption)': scores_skf,
    'GroupKFold      (no user leakage)': scores_gkf,
    'TimeSeriesSplit (no future info) ': scores_tss,
})
print(summary.round(4))
print('\nMean PR-AUC by strategy:')
print(summary.mean().round(4))

## What to read in the output

- **StratifiedKFold** is usually the most optimistic — IID assumption is rarely true.
- **GroupKFold** comes down when same-user signal in train was helping test scores.
- **TimeSeriesSplit** is typically the most pessimistic (and most honest) — it forces the model to predict the future from the past.

**The number to trust** for a production launch decision is the one whose split mirrors how the model will actually be used. For fraud, that's almost always time-based + grouped.

## Bonus: spotting leakage by score gap

Rule of thumb: if your random-IID CV score is materially higher than your time-based score, you're probably leaking. The bigger the gap, the bigger the leak.